# SGE Oligo Library Generator

This notebook generates oligonucleotide libraries for Saturation Genome Editing (SGE) experiments. Given a set of SGE oligos and AMP F/R primers, it produces two complementary libraries:

- **SNV library** — all single-nucleotide variants and 3-bp deletions
- **Full saturation mutagenesis library** — all possible amino acid substitutions at every codon (missense, synonymous, stop, and 3-nt deletions) for regions with `Library type = SNVs+3bpdel+Saturation`

For saturation libraries, reading frame and amino acid offset are derived automatically by fetching the gene's canonical CDS from the Ensembl REST API and matching the oligo's exonic sequence against it. Oligos may include intronic flanking sequence — the exonic portion is identified automatically.

Fixed edits (PAM edits, HDR markers) should be encoded as **lowercase bases** in the oligo sequence. They are preserved in all output oligos but excluded from mutagenesis.

---

## Input File Format

The input is an Excel file (`.xlsx`). Columns are identified by name and may appear in any order. Extra columns (e.g. `Gene`, `Exon`, `Length`) are ignored.

### Required columns

| Column | Description |
|---|---|
| `Oligo Name` | Name of the oligo — must follow the `{GENE}_{REGION}_*` naming convention (see below) |
| `Oligo Sequence` | DNA sequence. **Lowercase bases** denote fixed edits (PAM edits, HDR markers) that are preserved in all output oligos but excluded from mutagenesis |

### Optional but recommended columns

| Column | Description |
|---|---|
| `Type` | Role of the oligo: `AMP Forward`, `AMP Reverse`, or `SGEoligo`. If absent, the role is inferred from `Oligo Name` (looks for `AMP_F`, `AMP_R`, or `sgeoligo` in the name) |

### Required on `SGEoligo` rows only

| Column | Description |
|---|---|
| `Library type` | `SNVs+3bpdel` — generate SNVs and 3-bp deletions only; `SNVs+3bpdel+Saturation` — also generate a full saturation mutagenesis library for this region |

### Optional / informational columns

| Column | Description |
|---|---|
| `Gene`, `Exon`, `Length` | Descriptive metadata — not used by the code |
| `sgRNA` rows | Any row with `Type = sgRNA` is silently ignored |

### Naming convention for `Oligo Name`

The first two underscore-delimited fields (`{GENE}_{REGION}`) define the region group. All rows sharing the same `{GENE}_{REGION}` prefix are processed together. Each region requires exactly one `AMP Forward`, one `AMP Reverse`, and one `SGEoligo` row:

```
{GENE}_{REGION}_*   — AMP Forward   : forward amplification primer (5'→3', forward strand)
{GENE}_{REGION}_*   — AMP Reverse   : reverse amplification primer (5'→3', reverse strand — will be reverse-complemented)
{GENE}_{REGION}_*   — SGEoligo      : HDR template tile sequence
```

For example: `PIK3CA_X16_AMP_F`, `PIK3CA_X16_AMP_R`, `PIK3CA_X16_sgeoligo`

> **Note:** sgRNA rows may use a slightly different region suffix (e.g. `PIK3CA_X9A_sgRNA1_W` alongside `PIK3CA_X9_*`). These will form their own group with no SGEoligo and be silently skipped.

### Notes
- Oligos may include intronic sequence at either or both ends — the exonic portion is identified automatically via the Ensembl CDS.
- Only isolated lowercase bases (≤ 2 consecutive) are treated as fixed edits. Longer lowercase runs are assumed to be intronic or other non-coding sequence and will be auto-uppercased with a warning.
- Saturation mutagenesis regions require a network connection to the Ensembl REST API at runtime to fetch the gene's CDS.

In [33]:
import pandas as pd
from Bio.Seq import Seq
from datetime import datetime
import requests
import re

In [34]:
input_file = '/Users/ivan/Downloads/20260317_SGE_All_Oligos_merged_recoded.xlsx'
output_folder = '/Users/ivan/Desktop/'

file_name = f'CZI+CAVA_SGE_AllOligos_recoded'
oligos_in = pd.read_excel(input_file)

# Map 'Library type' column → 'full_mut' flag used downstream.
# 'SNVs+3bpdel+Saturation' → full saturation mutagenesis ('y'
# 'SNVs+3bpdel'            → SNV + 3-bp deletions only   ('')
if 'Library type' in oligos_in.columns:
    oligos_in['full_mut'] = oligos_in['Library type'].apply(
        lambda x: 'y' if isinstance(x, str) and 'Saturation' in x else ''
    )

oligos_in['region_key'] = oligos_in['Oligo Name'].apply(lambda x: '_'.join(x.strip().split('_')[:2]))

print(f"Loaded {len(oligos_in)} oligos")
print("Columns:", oligos_in.columns.tolist())
oligos_in.head()

Loaded 243 oligos
Columns: ['Gene', 'Exon', 'Oligo Name', 'Type', 'Oligo Sequence', 'Length', 'Library type', 'full_mut', 'region_key']


,Gene,Exon,Oligo Name,Type,Oligo Sequence,Length,Library type,full_mut,region_key
0,BRIP1,X10,BRIP1_X10_AMP_F,AMP Forward,GATGATACTGGTTGACACAA,20,NaN,,BRIP1_X10
1,BRIP1,X10,BRIP1_X10_AMP_R,AMP Reverse,CGTTTCACAGGTAGAAAAA,19,NaN,,BRIP1_X10
2,BRIP1,X10,BRIP1_X10_sgeoligo,SGEoligo,gTGTTTTATTTTATTTCAGTTGGTTAGAAGCAAACGCTGAATATCT...,161,SNVs+3bpdel,,BRIP1_X10
3,BRIP1,X11A,BRIP1_X11A_AMP_F,AMP Forward,CCCTCCCAACCCCTCTATAC,20,NaN,,BRIP1_X11A
4,BRIP1,X11A,BRIP1_X11A_AMP_R,AMP Reverse,ACCTGCTATTTTGCCTAAAAAGAT,24,NaN,,BRIP1_X11A


In [35]:
# Include all rows for regions that have a saturation-flagged sgeoligo,
# so that AMP_F and AMP_R rows (which have no 'Library type' value) are retained.
saturation_regions = oligos_in.loc[oligos_in['full_mut'] == 'y', 'region_key'].unique()
full_mut_oligos_in = oligos_in[oligos_in['region_key'].isin(saturation_regions)]
print(f"Loaded {len(full_mut_oligos_in)} full mutation oligos")
print("Columns:", full_mut_oligos_in.columns.tolist())
full_mut_oligos_in.head()

Loaded 27 full mutation oligos
Columns: ['Gene', 'Exon', 'Oligo Name', 'Type', 'Oligo Sequence', 'Length', 'Library type', 'full_mut', 'region_key']


,Gene,Exon,Oligo Name,Type,Oligo Sequence,Length,Library type,full_mut,region_key
183,PIK3CA,X16,PIK3CA_X16_AMP_F,AMP Forward,GCAAAGGTACCTAGTAAAGTTTTTAAC,27,NaN,,PIK3CA_X16
184,PIK3CA,X16,PIK3CA_X16_AMP_R,AMP Reverse,CTAAATTCATGCATCATAAGCTC,23,NaN,,PIK3CA_X16
185,PIK3CA,X16,PIK3CA_X16_sgeoligo,SGEoligo,TATTTTAAAGGCTTGAAGAGTGTCGAATTATGTCCTCTGCAAAAAG...,150,SNVs+3bpdel+Saturation,y,PIK3CA_X16
186,PIK3CA,X17,PIK3CA_X17_AMP_F,AMP Forward,AAATTTGCATCTGTGGCATT,20,NaN,,PIK3CA_X17
187,PIK3CA,X17,PIK3CA_X17_AMP_R,AMP Reverse,TTTCAACATACAGGTTGCCTTA,22,NaN,,PIK3CA_X17


In [36]:
codons_ranked_by_usage = {
    "A": ["GCC", "GCT", "GCA", "GCG"],
    "C": ["TGC", "TGT"],
    "D": ["GAC", "GAT"],
    "E": ["GAG", "GAA"],
    "F": ["TTC", "TTT"],
    "G": ["GGC", "GGA", "GGG", "GGT"],
    "H": ["CAC", "CAT"],
    "I": ["ATC", "ATT", "ATA"],
    "K": ["AAG", "AAA"],
    "L": ["CTG", "CTC", "CTT", "TTG", "TTA", "CTA"],
    "M": ["ATG"],
    "N": ["AAC", "AAT"],
    "P": ["CCC", "CCT", "CCA", "CCG"],
    "Q": ["CAG", "CAA"],
    "R": ["CGG", "AGA", "AGG", "CGC", "CGA", "CGT"],
    "S": ["AGC", "TCC", "TCT", "AGT", "TCA", "TCG"],
    "T": ["ACC", "ACA", "ACT", "ACG"],
    "V": ["GTG", "GTC", "GTT", "GTA"],
    "W": ["TGG"],
    "Y": ["TAC", "TAT"],
}


def extract_orf(tile, frame):
    """
    Extract the ORF from a tile sequence given the reading frame.

    Parameters
    ----------
    tile : str
        DNA tile sequence.
    frame : int or str
        Base position within a codon at which the tile starts (1, 2, or 3).
        1 = tile starts at the 1st base of a codon
        2 = tile starts at the 2nd base of a codon
        3 = tile starts at the 3rd (last) base of a codon

    Returns
    -------
    orf_nt : str
        Nucleotide sequence trimmed to complete codons, ending after the first
        stop codon (if present).
    orf_aa : str
        Translated amino acid sequence, ending after the first stop codon
        (if present).
    """
    frame = int(frame)
    if frame not in (1, 2, 3):
        raise ValueError(f"frame must be 1, 2, or 3; got {frame}")

    offset = (4 - frame) % 3
    orf_nt = tile[offset:]

    remainder = len(orf_nt) % 3
    if remainder:
        orf_nt = orf_nt[:-remainder]

    orf_aa = str(Seq(orf_nt).translate())

    stop_idx = orf_aa.find('*')
    if stop_idx != -1:
        orf_aa = orf_aa[:stop_idx + 1]
        orf_nt = orf_nt[:(stop_idx + 1) * 3]

    return orf_nt, orf_aa

In [37]:
def mutagenize(my_string):
    """Generate all SNVs, skipping lowercase (PAM) positions."""
    all_variants = []
    for i in range(len(my_string)):
        if my_string[i].isupper():
            for j in ("A", "C", "G", "T"):
                if j != my_string[i]:
                    all_variants.append(my_string[:i] + j + my_string[i+1:])
    return all_variants


In [38]:
def three_bp_del_mutagenize(my_string):
    """Generate all unique 3bp deletion variants."""
    upper_string = my_string.upper()
    del_size = 3
    all_del_variants = []
    for i in range(len(upper_string) - del_size + 1):
        all_del_variants.append(upper_string[:i] + upper_string[i+del_size:])
    return list(set(all_del_variants))

In [39]:
def rev_compl(st):
    """Return the reverse complement of a DNA sequence."""
    nn = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return "".join(nn[n] for n in reversed(st.upper()))

In [40]:
def make_snvs(oligos):
    # Generate all mutant oligos for each region
    mutations_out = []

    for region, group in oligos_in.groupby('region_key', sort=False):
        gene, regionname = region.split('_', 1)

        ampf = ampr = None
        sge_oligos = []

        has_type_col = 'Type' in group.columns
        for _, row in group.iterrows():
            name = row['Oligo Name']
            seq = row['Oligo Sequence']
            if has_type_col:
                t = str(row.get('Type', ''))
                is_ampf = t == 'AMP Forward'
                is_ampr = t == 'AMP Reverse'
                is_sge  = t == 'SGEoligo'
            else:
                is_ampf = 'AMP_F' in name
                is_ampr = 'AMP_R' in name
                is_sge  = 'sgeoligo' in name.lower()
            if is_ampf:
                ampf = seq.upper()
            elif is_ampr:
                ampr = rev_compl(seq)
            elif is_sge:
                sge_oligos.append(seq)

        if not sge_oligos:
            continue  # no sgeoligo in this group (e.g. sgRNA-only row with a different region suffix)
        if None in (ampf, ampr):
            print(f"Warning: missing AMP primers for region {region}")
            continue

        snv_counter = 0
        del_counter = 0
        for sge_seq in sge_oligos:
            # Uppercase long lowercase runs (intronic sequence) so they are
            # treated as mutable; single lowercase fixed-edit bases are preserved.
            sge_seq = _sanitize_tile_case(sge_seq, region)
            for mutant in mutagenize(sge_seq):
                snv_counter += 1
                mutations_out.append((f"{gene}_{regionname}_snv_{snv_counter}", ampf + mutant + ampr))
            for deletion in three_bp_del_mutagenize(sge_seq):
                del_counter += 1
                mutations_out.append((f"{gene}_{regionname}_del_{del_counter}", ampf + deletion + ampr))

    print(f"Total SNV/3-bp del oligos generated: {len(mutations_out)}")
    
    return mutations_out

In [41]:
def fetch_gene_data(gene_symbol: str, species: str = "homo_sapiens") -> dict:
    """
    Fetch the CDS, genomic coordinates, strand, and exon intervals for a
    gene's canonical transcript via Ensembl REST API.

    Returns
    -------
    dict with keys:
        cds            : str  — full coding sequence
        chrom          : str  — chromosome name (e.g. '3')
        strand         : int  — +1 or -1
        gene_start     : int  — 1-based +strand genomic start of the gene
        gene_end       : int  — 1-based +strand genomic end of the gene
        exon_intervals : list — sorted list of (start, end) 1-based inclusive,
                                +strand coords, for the canonical transcript
    """
    r = requests.get(
        f"https://rest.ensembl.org/lookup/symbol/{species}/{gene_symbol}",
        params={"expand": 1, "content-type": "application/json"}
    )
    r.raise_for_status()
    data = r.json()

    transcripts = data.get("Transcript", [])
    if not transcripts:
        raise ValueError(f"No transcripts found for gene: {gene_symbol}")

    canonical = next((t for t in transcripts if t.get("is_canonical")), None)
    transcript = canonical if canonical else transcripts[0]
    transcript_id = transcript["id"]
    print(f"  Using transcript {transcript_id} for {gene_symbol}" +
          (" (canonical)" if canonical else " (first available, no canonical found)"))

    r2 = requests.get(
        f"https://rest.ensembl.org/sequence/id/{transcript_id}",
        params={"type": "cds", "content-type": "text/plain"}
    )
    r2.raise_for_status()
    cds = r2.text.strip()

    chrom      = data["seq_region_name"]
    strand     = data["strand"]
    gene_start = data["start"]
    gene_end   = data["end"]
    exon_intervals = sorted(
        (e["start"], e["end"]) for e in transcript.get("Exon", [])
    )

    return {
        "cds":            cds,
        "chrom":          chrom,
        "strand":         strand,
        "gene_start":     gene_start,
        "gene_end":       gene_end,
        "exon_intervals": exon_intervals,
    }


def fetch_genomic_sequence(chrom: str, start: int, end: int, strand: int,
                           species: str = "homo_sapiens") -> str:
    """
    Fetch a region of genomic sequence oriented to the coding strand.

    For + strand genes: geno_seq[i] corresponds to genomic position start + i.
    For - strand genes: geno_seq[i] corresponds to genomic position end - i
    (the sequence is reverse-complemented so index 0 is the 5'-most coding base).

    Parameters
    ----------
    chrom  : chromosome name (e.g. '3')
    start  : 1-based +strand genomic start of the region to fetch
    end    : 1-based +strand genomic end of the region to fetch
    strand : +1 or -1

    Returns
    -------
    str — uppercase genomic sequence in coding-strand orientation
    """
    strand_param = -1 if strand == -1 else 1
    r = requests.get(
        f"https://rest.ensembl.org/sequence/region/{species}/{chrom}:{start}..{end}:{strand_param}",
        params={"content-type": "text/plain"}
    )
    r.raise_for_status()
    return r.text.strip().upper()


def _tile_to_regex(segment: str) -> str:
    """
    Convert a tile segment to a CDS search pattern.
    Uppercase bases must match exactly; lowercase bases (fixed edits such as
    PAM disruptions or synonymous markers) match any base via '.'.
    """
    return ''.join('.' if c.islower() else c for c in segment)


def _sanitize_tile_case(tile: str, region: str, max_lc_run: int = 2) -> str:
    """
    Warn and auto-correct runs of consecutive lowercase bases longer than
    max_lc_run. Fixed edits should always be isolated SNVs (single lowercase
    bases); longer runs are assumed to be formatting errors and are uppercased.

    Parameters
    ----------
    tile : str
        Input tile sequence.
    region : str
        Region name, used in warning messages.
    max_lc_run : int
        Maximum allowed run of consecutive lowercase bases. Runs strictly
        longer than this are uppercased. Default: 2.

    Returns
    -------
    str
        Tile with any over-length lowercase runs uppercased.
    """
    pattern = rf'[a-z]{{{max_lc_run + 1},}}'
    runs = re.findall(pattern, tile)
    if runs:
        longest = max(runs, key=len)
        print(f"  WARNING ({region}): {len(runs)} lowercase run(s) longer than "
              f"{max_lc_run}bp detected (longest: '{longest}', {len(longest)}bp). "
              f"These have been auto-uppercased. Only fixed-edit SNVs should be "
              f"lowercase — check that the SGE oligo sequence is correctly formatted.")
        tile = re.sub(pattern, lambda m: m.group(0).upper(), tile)
    return tile


def get_exonic_tile_info(cds: str, tile: str, amp_f: str = None, amp_r_rc: str = None,
                         region: str = "", max_lc_run: int = 2,
                         genomic_ctx: dict = None):
    """
    Find the exonic portion of a tile, then compute reading frame and AA offset.

    Detection strategy (tried in order until one succeeds):

    1. Genomic anchor (primary): requires both AMP primers and a genomic_ctx dict
       containing the coding-strand genomic sequence and exon intervals.
       Finds each primer in the genomic sequence to determine the tile's exact
       genomic span, then intersects with exon intervals to get precise intronic
       prefix/suffix lengths.  Works whether primers are intronic or exonic.

    2. CDS primer anchor: uses the longest suffix of AMP_F / prefix of AMP_R_RC
       that matches the CDS to identify exon boundaries.  Only activates when
       at least one primer crosses into an exon.

    3. Middle-tile anchor: uses a 30 bp segment from the tile's middle (reliably
       exonic) to pin the correct CDS position, then scans outward.  Immune to
       CDS repeats near tile ends.

    4. Binary search fallback: searches for the longest CDS-matching substring.
       Least reliable; used only when all other methods fail.

    Parameters
    ----------
    cds : str
        Full coding sequence of the gene.
    tile : str
        Tile sequence. Lowercase bases are fixed edits relative to the reference.
    amp_f : str, optional
        AMP Forward primer (uppercase, coding-strand orientation).
    amp_r_rc : str, optional
        AMP Reverse primer reverse-complemented to the coding strand (uppercase).
    region : str
        Region name, used in warning messages.
    max_lc_run : int
        Lowercase runs longer than this are auto-uppercased before matching.
    genomic_ctx : dict, optional
        Dict with keys:
            geno_seq          : str   — coding-strand genomic sequence
            geno_region_start : int   — 1-based +strand start of geno_seq[0]
                                        (used for + strand genes)
            geno_region_end   : int   — 1-based +strand end of geno_seq[-1]
                                        (used for - strand genes)
            strand            : int   — +1 or -1
            exon_intervals    : list  — sorted (start, end) 1-based inclusive,
                                        +strand genomic coords

    Returns
    -------
    exonic : str
        The exonic portion of the tile (original case preserved).
    intron_prefix : str
        Intronic bases at the 5' end of the tile.
    intron_suffix : str
        Intronic bases at the 3' end of the tile.
    frame : int
        Reading frame (1/2/3) of the exonic portion's start in the CDS.
    aa_offset : int
        1-based AA position of the first complete in-frame codon in the tile.
    """
    lc_pattern = rf'[a-z]{{{max_lc_run + 1},}}'
    lc_runs = [(m.start(), m.end()) for m in re.finditer(lc_pattern, tile)]

    tile = _sanitize_tile_case(tile, region, max_lc_run)
    cds_upper = cds.upper()

    def match_start(segment):
        """Return 0-based CDS position of match, or -1 if not found."""
        m = re.search(_tile_to_regex(segment), cds_upper)
        return m.start() if m else -1

    best_start = best_end = None

    # ── 1. Genomic anchor ─────────────────────────────────────────────────────
    if amp_f is not None and amp_r_rc is not None and genomic_ctx is not None:
        geno_seq          = genomic_ctx["geno_seq"]
        geno_region_start = genomic_ctx["geno_region_start"]
        geno_region_end   = genomic_ctx["geno_region_end"]
        strand            = genomic_ctx["strand"]
        exon_intervals    = genomic_ctx["exon_intervals"]

        amp_f_m = re.search(re.escape(amp_f.upper()), geno_seq)
        amp_r_m = re.search(re.escape(amp_r_rc.upper()), geno_seq)

        if amp_f_m and amp_r_m:
            idx_start = amp_f_m.end()    # 0-based index in geno_seq of tile[0]
            idx_end   = amp_r_m.start()  # 0-based index in geno_seq of tile[-1]+1

            if 0 <= idx_start < idx_end:
                if strand == 1:
                    tile_geno_start = geno_region_start + idx_start
                    tile_geno_end   = geno_region_start + idx_end - 1
                    overlapping = [(s, e) for s, e in exon_intervals
                                   if s <= tile_geno_end and e >= tile_geno_start]
                    if overlapping:
                        if len(overlapping) > 1:
                            print(f"  Note ({region}): tile spans {len(overlapping)} exons; "
                                  f"using outermost exonic boundaries.")
                        best_start = max(0, overlapping[0][0] - tile_geno_start)
                        best_end   = len(tile) - max(0, tile_geno_end - overlapping[-1][1])
                else:  # strand == -1
                    # geno_seq[i] corresponds to genomic position geno_region_end - i
                    tile_geno_top = geno_region_end - idx_start
                    tile_geno_bot = geno_region_end - (idx_end - 1)
                    overlapping = [(s, e) for s, e in exon_intervals
                                   if s <= tile_geno_top and e >= tile_geno_bot]
                    if overlapping:
                        if len(overlapping) > 1:
                            print(f"  Note ({region}): tile spans {len(overlapping)} exons; "
                                  f"using outermost exonic boundaries.")
                        # For - strand: exon with highest coords is 5' (first in coding dir)
                        best_start = max(0, tile_geno_top - overlapping[-1][1])
                        best_end   = len(tile) - max(0, overlapping[0][0] - tile_geno_bot)

        # Verify the detected exonic region is actually in the CDS.
        # If not (e.g. tile extends into 5' or 3' UTR within the last/first exon),
        # trim UTR bases from the ends with a short binary search.
        if best_start is not None and match_start(tile[best_start:best_end]) == -1:
            # Trim right: longest prefix of tile[best_start:] that matches CDS.
            lo, hi, last_good = best_start + 1, best_end, best_start
            while lo <= hi:
                mid = (lo + hi) // 2
                if match_start(tile[best_start:mid]) != -1:
                    last_good = mid; lo = mid + 1
                else:
                    hi = mid - 1
            best_end = last_good
            # Trim left: longest suffix of tile[:best_end] that matches CDS.
            if match_start(tile[best_start:best_end]) == -1:
                lo, hi, last_good = best_start, best_end - 1, best_end
                while lo <= hi:
                    mid = (lo + hi) // 2
                    if match_start(tile[mid:best_end]) != -1:
                        last_good = mid; hi = mid - 1
                    else:
                        lo = mid + 1
                best_start = last_good
        if best_start is None or match_start(tile[best_start:best_end]) == -1:
            if best_start is not None:
                print(f"  Warning ({region}): genomic anchor: AMP primers found but "
                      f"could not resolve CDS boundaries; falling back.")
            else:
                print(f"  Warning ({region}): genomic anchor failed "
                      f"(AMP primers not found in genomic sequence or no overlapping exon); "
                      f"falling back to sequence-based detection.")
            best_start = best_end = None

    # ── 2. CDS primer anchor ──────────────────────────────────────────────────
    if best_start is None and amp_f is not None and amp_r_rc is not None:
        cds_tile_start = None
        for n in range(min(len(amp_f), 20), 9, -1):
            m = re.search(re.escape(amp_f[-n:].upper()), cds_upper)
            if m:
                cds_tile_start = m.end()
                break

        cds_tile_end = None
        for n in range(min(len(amp_r_rc), 20), 9, -1):
            m = re.search(re.escape(amp_r_rc[:n].upper()), cds_upper)
            if m:
                cds_tile_end = m.start()
                break

        if cds_tile_start is not None and cds_tile_end is not None and cds_tile_start < cds_tile_end:
            ANC = 12

            expected_5 = cds_upper[cds_tile_start:cds_tile_start + ANC]
            for i in range(min(len(tile) - ANC + 1, 50)):
                if re.fullmatch(_tile_to_regex(tile[i:i + ANC]), expected_5):
                    best_start = i
                    break

            expected_3 = cds_upper[cds_tile_end - ANC:cds_tile_end]
            for j in range(len(tile), max(len(tile) - 50, ANC - 1), -1):
                if re.fullmatch(_tile_to_regex(tile[j - ANC:j]), expected_3):
                    best_end = j
                    break

        if best_start is None or best_end is None:
            best_start = best_end = None

    # ── 3. Middle-tile anchor ─────────────────────────────────────────────────
    if best_start is None:
        ANC_M = 30
        ANC   = 12
        mid_pos  = len(tile) // 2
        mid_seg  = tile[mid_pos - ANC_M // 2:mid_pos + ANC_M // 2]
        mid_cds  = match_start(mid_seg)

        if mid_cds != -1:
            cds_est_start = mid_cds - (mid_pos - ANC_M // 2)
            cds_est_end   = mid_cds + (len(tile) - (mid_pos - ANC_M // 2))

            ms = lc_runs[0][1] if lc_runs and lc_runs[0][0] <= 1 else 0
            me = lc_runs[-1][0] if lc_runs and lc_runs[-1][1] == len(tile) else len(tile)

            found_start = found_end = False

            if cds_est_start >= 0:
                exp5 = cds_upper[cds_est_start:cds_est_start + ANC]
                for i in range(ms, min(ms + 50, len(tile) - ANC + 1)):
                    if re.fullmatch(_tile_to_regex(tile[i:i + ANC]), exp5):
                        best_start = i
                        found_start = True
                        break

            if cds_est_end <= len(cds_upper):
                exp3 = cds_upper[cds_est_end - ANC:cds_est_end]
                for j in range(me, max(me - 50, ANC - 1), -1):
                    if re.fullmatch(_tile_to_regex(tile[j - ANC:j]), exp3):
                        best_end = j
                        found_end = True
                        break

            if not (found_start and found_end):
                best_start = best_end = None

    # ── 4. Binary search fallback ─────────────────────────────────────────────
    if best_start is None:
        search_start = lc_runs[0][1] if lc_runs and lc_runs[0][0] <= 1 else 0
        search_end   = lc_runs[-1][0] if lc_runs and lc_runs[-1][1] == len(tile) else len(tile)

        best_start, best_end = search_start, search_start
        for i in range(search_start, search_end):
            lo, hi = i + 1, search_end
            last_good = i
            while lo <= hi:
                mid = (lo + hi) // 2
                if match_start(tile[i:mid]) != -1:
                    last_good = mid
                    lo = mid + 1
                else:
                    hi = mid - 1
            if (last_good - i) > (best_end - best_start):
                best_start, best_end = i, last_good

    exonic = tile[best_start:best_end]
    intron_prefix = tile[:best_start]
    intron_suffix = tile[best_end:]

    if len(exonic) < 10:
        raise ValueError(
            f"No substantial exonic sequence found in tile (longest CDS match: {len(exonic)}bp). "
            "Check that the SGE oligo sequence and gene symbol are correct."
        )

    nt_pos = match_start(exonic) + 1           # 1-based CDS position
    frame = ((nt_pos - 1) % 3) + 1
    skip = (4 - frame) % 3                     # bases to first complete codon
    aa_offset = (nt_pos + skip - 1) // 3 + 1  # 1-based AA of first full codon

    return exonic, intron_prefix, intron_suffix, frame, aa_offset


In [42]:
def make_mutations(region_name,
                   region,
                   region_flanks=[Seq(''), Seq('')],
                   nt_start=0,
                   wt_only=False,
                   incl_wt = False,
                   synonymous=True,
                   stops='TAA',
                   all3ntdeletions=True,
                   mutation_list=False,
                   codons_ranked_by_usage=codons_ranked_by_usage,
                   aa_start=0):

    oligo_array = {}

    if (len(region) / 3 != len(region) // 3) | (nt_start / 3 != nt_start // 3):
        print('Region is not translatable!')

    else:

        if incl_wt:
            oligo_name = region_name + '_WT'
            wt_seq = region_flanks[0] + region + region_flanks[1]
            oligo_array[oligo_name] = wt_seq

        if not wt_only:

            if mutation_list == False:

                seen_del_seqs = set()

                for j in range(0, len(region), 3):

                    aa = region[j:(j + 3)].upper().translate()
                    for aa_to in codons_ranked_by_usage.keys():
                        if aa_to != aa:
                            oligo_name = region_name + '_' + str(aa) + str((nt_start + j) // 3 + 1 + aa_start) + str(aa_to)
                            seq_to_append = \
                                region_flanks[0] + \
                                region[0:j] + Seq(codons_ranked_by_usage[aa_to][0]) + \
                                region[(j + 3):] + \
                                region_flanks[1]
                            oligo_array[oligo_name] = seq_to_append

                    if synonymous:
                        if len(codons_ranked_by_usage[str(aa)]) > 1:
                            oligo_name = region_name + '_' + str(aa) + str((nt_start + j) // 3 + 1 + aa_start) + str(aa)
                            possible_codons = codons_ranked_by_usage[str(aa)].copy()
                            possible_codons.remove(str(region[j:(j + 3)]).upper())
                            seq_to_append = \
                                region_flanks[0] + \
                                region[0:j] + Seq(possible_codons[0]) + \
                                region[(j + 3):] + \
                                region_flanks[1]
                            oligo_array[oligo_name] = seq_to_append

                    if stops:
                        oligo_name = region_name + '_' + str(aa) + str((nt_start + j) // 3 + 1 + aa_start) + 'X'
                        seq_to_append = \
                            region_flanks[0] + \
                            region[0:j] + Seq(stops) + \
                            region[(j + 3):] + \
                            region_flanks[1]
                        oligo_array[oligo_name] = seq_to_append

                    if all3ntdeletions:
                        for k in range(0, 3):
                            if j + k + 3 <= len(region):
                                seq_to_append = \
                                    region_flanks[0] + \
                                    region[0:(j + k)] + \
                                    region[(j + k + 3):] + \
                                    region_flanks[1]
                                seq_str = str(seq_to_append)
                                if seq_str not in seen_del_seqs:
                                    seen_del_seqs.add(seq_str)
                                    oligo_name = region_name + '_' + 'del' + str(nt_start + j + k + 1 + 3 * aa_start)
                                    oligo_array[oligo_name] = seq_to_append

            else:

                for i in range(len(mutation_list)):
                    oligo_name = region_name + '_' + 'variant' + str(i + 1)
                    seq_to_append = region
                    for k, v in enumerate(mutation_list[i]):
                        aa_from = v[0]
                        aa_to = v[-1]
                        pos = int(v[1:-1])
                        j = 3 * (pos - (nt_start // 3 + 1))
                        aa = region[j:(j + 3)].upper().translate()
                        if aa != aa_from:
                            print('Check mutation list!')
                        else:
                            seq_to_append = \
                                seq_to_append[0:j] + \
                                Seq(codons_ranked_by_usage[aa_to][0]) + \
                                seq_to_append[(j + 3):]
                    seq_to_append = region_flanks[0] + seq_to_append + region_flanks[1]
                    oligo_array[oligo_name] = seq_to_append

    return oligo_array

In [43]:
def make_tile_mutations(tile_name,
                        tile,
                        tile_amp_f,
                        tile_amp_r,
                        frame,
                        nt_start=0,
                        wt_only=False,
                        synonymous=True,
                        stops='TAA',
                        all3ntdeletions=False,
                        mutation_list=False,
                        codons_ranked_by_usage=codons_ranked_by_usage,
                        aa_start=0):
    """
    Generate a full saturation mutagenesis oligo library for a single tile.

    Extracts the in-frame ORF from the tile, then generates all variants
    (missense, synonymous, stop, 3-nt deletions) using make_mutations.
    Each oligo is returned as: tile_amp_f + variant_tile + tile_amp_r.

    Lowercase bases in the tile are preserved in non-mutagenized positions
    to demarcate fixed edits; only the substituted codon is uppercased
    (as it comes from codons_ranked_by_usage).
    """
    orf_nt, orf_aa = extract_orf(tile.upper(), frame)
    offset = (4 - int(frame)) % 3

    orf_nt_original_case = tile[offset:offset + len(orf_nt)]
    tile_suffix = tile[offset + len(orf_nt):]

    if orf_aa.endswith('*'):
        tile_suffix = orf_nt_original_case[-3:] + tile_suffix
        orf_nt_original_case = orf_nt_original_case[:-3]

    tile_prefix = tile[:offset]

    region_flanks = [
        Seq(tile_amp_f) + Seq(tile_prefix),
        Seq(tile_suffix) + Seq(tile_amp_r),
    ]

    return make_mutations(
        region_name=tile_name,
        region=Seq(orf_nt_original_case),
        region_flanks=region_flanks,
        nt_start=nt_start,
        wt_only=wt_only,
        synonymous=synonymous,
        stops=stops,
        all3ntdeletions=all3ntdeletions,
        mutation_list=mutation_list,
        codons_ranked_by_usage=codons_ranked_by_usage,
        aa_start=aa_start,
    )

In [44]:
def get_full_mut_data(full_mut_oligos_in, preview_n=5):
    """
    Generate full saturation mutagenesis oligo libraries for all tiles
    marked full_mut='y' in the input DataFrame.

    For each gene, fetches the CDS and coding-strand genomic sequence from
    Ensembl (once per gene, cached across regions).  Uses AMP primer positions
    in the genomic sequence together with exon intervals to precisely determine
    each tile's exonic boundaries, reading frame, and AA offset.

    Parameters
    ----------
    full_mut_oligos_in : pd.DataFrame
        Subset of oligos_in where full_mut == 'y'. Must contain columns:
        'Oligo Name', 'Oligo Sequence', 'region_key'.
    preview_n : int
        Number of oligos to print per region for spot-checking.

    Returns
    -------
    all_oligos : dict
        Dict mapping oligo names to Seq objects for all regions.
    """
    all_oligos = {}
    gene_cache = {}  # gene_symbol -> (cds, genomic_ctx)

    for region, group in full_mut_oligos_in.groupby('region_key', sort=False):
        gene = region.split('_', 1)[0]
        print(f"Processing {region} (gene: {gene})...")

        ampf = ampr = tile = None
        has_type_col = 'Type' in group.columns
        for _, row in group.iterrows():
            name = row['Oligo Name']
            seq = row['Oligo Sequence']
            if has_type_col:
                t = str(row.get('Type', ''))
                is_ampf = t == 'AMP Forward'
                is_ampr = t == 'AMP Reverse'
                is_sge  = t == 'SGEoligo'
            else:
                is_ampf = 'AMP_F' in name
                is_ampr = 'AMP_R' in name
                is_sge  = 'sgeoligo' in name.lower()
            if is_ampf:
                ampf = seq.upper()
            elif is_ampr:
                ampr = rev_compl(seq)
            elif is_sge:
                tile = seq

        if None in (ampf, ampr, tile):
            print(f"  Warning: missing AMP Forward, AMP Reverse, or SGEoligo for {region} — skipping.")
            continue

        # Fetch gene data and genomic sequence once per gene (cached).
        if gene not in gene_cache:
            print(f"  Fetching Ensembl data for {gene}...")
            gene_data = fetch_gene_data(gene)
            print(f"  Fetching genomic sequence ({gene_data['chrom']}:{gene_data['gene_start']}-{gene_data['gene_end']}, "
                  f"strand={gene_data['strand']})...")
            geno_seq = fetch_genomic_sequence(
                gene_data["chrom"], gene_data["gene_start"], gene_data["gene_end"],
                gene_data["strand"]
            )
            genomic_ctx = {
                "geno_seq":          geno_seq,
                "geno_region_start": gene_data["gene_start"],
                "geno_region_end":   gene_data["gene_end"],
                "strand":            gene_data["strand"],
                "exon_intervals":    gene_data["exon_intervals"],
            }
            gene_cache[gene] = (gene_data["cds"], genomic_ctx)

        cds, genomic_ctx = gene_cache[gene]

        exonic, intron_prefix, intron_suffix, frame, aa_offset = get_exonic_tile_info(
            cds, tile, ampf, ampr, region, genomic_ctx=genomic_ctx
        )

        if intron_prefix or intron_suffix:
            print(f"  Tile is partially intronic: {len(intron_prefix)}bp 5' intron, {len(intron_suffix)}bp 3' intron trimmed.")

        print(f"  frame={frame}, aa_offset={aa_offset} (first complete codon)")

        adj_ampf = ampf + intron_prefix
        adj_ampr = intron_suffix + ampr

        oligos = make_tile_mutations(
            tile_name=region,
            tile=exonic,
            tile_amp_f=adj_ampf,
            tile_amp_r=adj_ampr,
            frame=frame,
            aa_start=aa_offset - 1,
        )

        # Rename oligos to {region}_{mutant}_{number}
        numbered = {
            f"{name}_{i + 1}": seq
            for i, (name, seq) in enumerate(oligos.items())
        }

        print(f"  Generated {len(numbered)} oligos for {region}. Preview:")
        for name, seq in list(numbered.items())[:preview_n]:
            print(f"    {name}: {seq}")

        all_oligos.update(numbered)

    print(f"\nTotal full-mut oligos generated: {len(all_oligos)}")
    return all_oligos


In [45]:
def check_output(mutations_out, preview_start=0, preview_end=10):
    for name, seq in mutations_out[preview_start:preview_end]:
        print(f"{name}\t{seq}") 

In [46]:
def write_oligos(oligo_array, filepath):
    """
    Write an oligo array to a .txt file.

    Each entry is written as:
        Sequence_Name
        sequence
        (blank line)

    Parameters
    ----------
    oligo_array : list of (name, seq) tuples
        Oligos to write.
    filepath : str
        Path to the output .txt file.
    """
    with open(filepath, 'w') as f:
        for name, seq in oligo_array:
            f.write(f"{name}\n{seq}\n\n")
    print(f"Wrote {len(oligo_array)} oligos to {filepath}")

In [47]:
def main(save=False):
    snv_libs = make_snvs(oligos_in)
    check_output(snv_libs)

    full_mut_libs = get_full_mut_data(full_mut_oligos_in)

    if save:
        combined = snv_libs + list(full_mut_libs.items())
        date_str = datetime.today().strftime('%Y%m%d')
        write_oligos(combined, f"{output_folder}{date_str}_{file_name}_oligos.txt")

In [48]:
main(save = True)

  WARNING (PIK3CA_X12B): 1 lowercase run(s) longer than 2bp detected (longest: 'taaaataatgtaaaatagtaaataatgtttaattacaataatat', 44bp). These have been auto-uppercased. Only fixed-edit SNVs should be lowercase — check that the SGE oligo sequence is correctly formatted.
  WARNING (PIK3CA_X14A): 1 lowercase run(s) longer than 2bp detected (longest: 'tatatatatatatttttaatttt', 23bp). These have been auto-uppercased. Only fixed-edit SNVs should be lowercase — check that the SGE oligo sequence is correctly formatted.
  WARNING (PIK3CA_X21A): 1 lowercase run(s) longer than 2bp detected (longest: 'cttattacttatag', 14bp). These have been auto-uppercased. Only fixed-edit SNVs should be lowercase — check that the SGE oligo sequence is correctly formatted.
Total SNV/3-bp del oligos generated: 42424
BRIP1_X10_snv_1	GATGATACTGGTTGACACAAgAGTTTTATTTTATTTCAGTTGGTTAGAAGCAAACGCTGAATATCTTGTAGAAAGAGATTATGAATCAGCTTGTAAAATATGGAGTGGAAATGAAATGCTCTTAACTTTACACAAAATGGGTATCACgACTGCTACTTTTCCCATTTTGCAGGTAAGATATTTTTTCT